In [ ]:
# Cell 1 — Drive mount + disease signature
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Ritschel_Research/copd_scrna'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

# COPD disease signature — alveolar niche focus (Sauler et al. 2022)
# AT2 epithelial: HHIP, NUPR1, SFTPC; macrophage: SPP1, MT2A, MMP9;
# endothelial: CXCL1, CXCL2; general COPD: SERPINA1, CCL2
DISEASE_GENES = [
    'HHIP', 'NUPR1', 'SFTPC', 'SFTPB',
    'SPP1', 'MT2A', 'MT1X', 'MMP9',
    'CXCL1', 'CXCL2', 'SERPINA1', 'CCL2', 'CCL3'
]
DISEASE = "COPD"
GEO_ID  = "GSE171541"
print("Disease signature defined:", DISEASE_GENES)

In [ ]:
# Cell 2 — Install dependencies
!pip install scvi-tools scanpy anndata mygene chembl_webresource_client --quiet

In [ ]:
# Cell 3 — Download files + build AnnData from dense text matrix
import subprocess, os, gzip
import pandas as pd
import numpy as np
import scipy.sparse as sp
import anndata
import scanpy as sc

sc.settings.verbosity = 3

BASE_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE171nnn/GSE171541/suppl"
UMI_GZ   = f"{DRIVE_DIR}/GSE171541_UMI_count.txt.gz"
META_GZ  = f"{DRIVE_DIR}/GSE171541_CellType_Metadata.txt.gz"
H5AD_RAW = f"{DRIVE_DIR}/copd_raw.h5ad"

# ── Download ──────────────────────────────────────────────────────────────────
for url, dest in [
    (f"{BASE_URL}/GSE171541_UMI_count.txt.gz",        UMI_GZ),
    (f"{BASE_URL}/GSE171541_CellType_Metadata.txt.gz", META_GZ),
]:
    if not os.path.exists(dest):
        print(f"Downloading {os.path.basename(dest)} ...")
        subprocess.run(['wget', '-q', '-O', dest, url], check=True)
    else:
        print(f"Already present: {os.path.basename(dest)}")

# ── Load or build AnnData ─────────────────────────────────────────────────────
if os.path.exists(H5AD_RAW):
    print("Loading cached raw h5ad ...")
    adata = sc.read_h5ad(H5AD_RAW)
else:
    # 1. Metadata (small — load fully)
    meta = pd.read_csv(META_GZ, sep='\t', index_col=0)
    print(f"Metadata: {meta.shape[0]} cells, columns: {meta.columns.tolist()}")

    # 2. Read UMI header to get cell barcodes
    with gzip.open(UMI_GZ, 'rt') as f:
        header = f.readline().rstrip('\n').split('\t')
    cell_ids = header[1:]  # 66,610 cell barcodes
    print(f"Cells in matrix: {len(cell_ids)}")

    # 3. Chunked load — read CHUNK_SIZE genes at a time → sparse
    CHUNK_SIZE = 500
    sparse_chunks = []
    gene_names    = []

    print("Reading UMI matrix in chunks ...")
    reader = pd.read_csv(
        UMI_GZ, sep='\t', index_col=0,
        chunksize=CHUNK_SIZE, dtype=np.float32
    )
    for i, chunk in enumerate(reader):
        gene_names.extend(chunk.index.tolist())
        sparse_chunks.append(sp.csr_matrix(chunk.values, dtype=np.float32))
        if i % 10 == 0:
            print(f"  chunk {i} — {len(gene_names)} genes processed ...")

    # 4. Stack (genes x cells), transpose → cells x genes
    print("Stacking sparse chunks ...")
    X = sp.vstack(sparse_chunks).T.tocsr()
    print(f"Matrix shape (cells x genes): {X.shape}")

    # 5. Build AnnData
    adata = anndata.AnnData(
        X   = X,
        obs = pd.DataFrame(index=cell_ids),
        var = pd.DataFrame(index=gene_names),
    )
    adata.obs_names.name = None
    adata.var_names.name = None

    # 6. Add condition from barcode prefix
    adata.obs['condition'] = adata.obs_names.str.extract(r'^(copd|ctl)')[0].values
    adata.obs['sample']    = adata.obs_names.str.extract(r'^([^_]+)')[0].values
    print("Condition counts:\n", adata.obs['condition'].value_counts())

    # 7. Save raw checkpoint
    adata.write_h5ad(H5AD_RAW)
    print(f"Raw h5ad saved -> {H5AD_RAW}")

print(adata)

In [ ]:
# Cell 4 — QC
QC_H5AD = f"{DRIVE_DIR}/copd_qc.h5ad"

if os.path.exists(QC_H5AD):
    adata = sc.read_h5ad(QC_H5AD)
    print("Loaded QC checkpoint.")
else:
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
    )

    print("Pre-filter:", adata.shape)
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_cells(adata, max_genes=6000)
    adata = adata[adata.obs['pct_counts_mt'] < 20].copy()
    sc.pp.filter_genes(adata, min_cells=3)
    print("Post-filter:", adata.shape)

    adata.write_h5ad(QC_H5AD)
    print(f"QC checkpoint saved -> {QC_H5AD}")

In [ ]:
# Cell 5 — Normalize + log1p + HVG
NORM_H5AD = f"{DRIVE_DIR}/copd_norm.h5ad"

if os.path.exists(NORM_H5AD):
    adata = sc.read_h5ad(NORM_H5AD)
    print("Loaded norm checkpoint.")
else:
    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(
        adata, n_top_genes=3000,
        subset=False, layer='counts',
        flavor='seurat_v3', batch_key='sample'
    )
    print(f"HVGs: {adata.var['highly_variable'].sum()}")

    adata.write_h5ad(NORM_H5AD)
    print(f"Norm checkpoint saved -> {NORM_H5AD}")

In [ ]:
# Cell 6 — scVI integration
import scvi
SCVI_H5AD = f"{DRIVE_DIR}/copd_scvi.h5ad"

if os.path.exists(SCVI_H5AD):
    adata = sc.read_h5ad(SCVI_H5AD)
    print("Loaded scVI checkpoint.")
else:
    scvi.model.SCVI.setup_anndata(
        adata, layer='counts',
        batch_key='sample',
        categorical_covariate_keys=['condition']
    )
    model = scvi.model.SCVI(adata, n_latent=30)
    model.train(max_epochs=150, accelerator='gpu')
    adata.obsm['X_scVI'] = model.get_latent_representation()
    model.save(f"{DRIVE_DIR}/copd_scvi_model", overwrite=True)

    adata.write_h5ad(SCVI_H5AD)
    print(f"scVI checkpoint saved -> {SCVI_H5AD}")

In [ ]:
# Cell 7 — Neighbors + UMAP
UMAP_H5AD = f"{DRIVE_DIR}/copd_umap.h5ad"

if os.path.exists(UMAP_H5AD):
    adata = sc.read_h5ad(UMAP_H5AD)
    print("Loaded UMAP checkpoint.")
else:
    sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=15)
    sc.tl.umap(adata)
    adata.write_h5ad(UMAP_H5AD)
    print(f"UMAP checkpoint saved -> {UMAP_H5AD}")

sc.pl.umap(adata, color=['condition', 'sample'], ncols=2)

In [ ]:
# Cell 8 — Leiden clustering
CLUST_H5AD = f"{DRIVE_DIR}/copd_clustered.h5ad"

if os.path.exists(CLUST_H5AD):
    adata = sc.read_h5ad(CLUST_H5AD)
    print("Loaded clustering checkpoint.")
else:
    sc.tl.leiden(
        adata, resolution=0.5,
        flavor='igraph', n_iterations=2, directed=False
    )
    print("Clusters:", adata.obs['leiden'].nunique())
    adata.write_h5ad(CLUST_H5AD)

sc.pl.umap(adata, color='leiden', legend_loc='on data')

In [ ]:
# Cell 9 — Dotplot + TOP_CLUSTERS + intentional stop
# COPD-relevant markers for annotation
# AT2: SFTPC, SFTPB, HHIP
# Macrophage: SPP1, MMP9, MT2A
# Endothelial: PECAM1, CLDN5
# T cell: CD3D, CD3E
# B cell: CD79A
# Monocyte: S100A8, S100A9
# Stromal: DCN, COL1A1
MARKERS = [
    'SFTPC','SFTPB','HHIP',        # AT2
    'AGER','CAV1',                  # AT1
    'SPP1','MMP9','MT2A','MT1X',    # macrophage
    'PECAM1','CLDN5',               # endothelial
    'CD3D','CD3E',                  # T cell
    'CD79A','MS4A1',                # B cell
    'S100A8','S100A9',              # monocyte
    'DCN','COL1A1',                 # stromal
    'TPSAB1','CPA3',                # mast
    'FOXJ1',                        # ciliated
]

existing = [g for g in MARKERS if g in adata.var_names]
sc.pl.dotplot(adata, var_names=existing, groupby='leiden',
              standard_scale='var', figsize=(14, 8))

# ── INSPECT DOTPLOT — then set TOP_CLUSTERS manually below ──────────────────
# TOP_CLUSTERS should be the macrophage/AT2/monocyte clusters most relevant
# to COPD (disease-enriched cell types from the paper: AT2, macrophages, cap-EC)
TOP_CLUSTERS = []   # <── fill in after dotplot inspection

print("Cluster sizes:\n", adata.obs['leiden'].value_counts().sort_index())

raise Exception(
    "STOP — inspect dotplot and cluster sizes above, "
    "set TOP_CLUSTERS, then continue from Cell 10."
)

In [ ]:
# Cell 10 — Wilcoxon DE (COPD vs control, within TOP_CLUSTERS)
import pandas as pd

# Subset to TOP_CLUSTERS for disease-relevant DE
adata_sub = adata[adata.obs['leiden'].isin([str(c) for c in TOP_CLUSTERS])].copy()
print(f"Cells in TOP_CLUSTERS: {adata_sub.shape[0]}")

# Use condition as groupby — COPD-upregulated genes are disease markers
sc.tl.rank_genes_groups(
    adata_sub, groupby='condition',
    groups=['copd'], reference='ctl',
    method='wilcoxon', layer='counts',
    pts=True
)

de = sc.get.rank_genes_groups_df(adata_sub, group='copd')
de = de[de['pvals_adj'] < 0.05].sort_values('scores', ascending=False)
print(f"Significant DE genes (COPD vs ctl): {len(de)}")
print(de.head(20))

de.to_csv(f"{DRIVE_DIR}/copd_DE_genes.csv", index=False)

In [ ]:
# Cell 11 — ChEMBL bioactivity screening
from chembl_webresource_client.new_client import new_client
import time

target_client   = new_client.target
activity_client = new_client.activity

PCHEMBL_THRESHOLD = 7.5
EXCLUDE = {'staurosporine', 'florbenazine'}

de_genes = de['names'].tolist()
print(f"Screening {len(de_genes)} DE genes against ChEMBL ...")

results = []
for gene in de_genes:
    targets = target_client.filter(
        target_synonym__iexact=gene,
        target_type='SINGLE PROTEIN',
        organism='Homo sapiens'
    ).only(['target_chembl_id','pref_name'])

    for t in targets:
        tid = t['target_chembl_id']
        acts = activity_client.filter(
            target_chembl_id=tid,
            standard_type__in=['IC50','Ki','Kd','EC50'],
            pchembl_value__isnull=False,
            assay_type='B'
        ).only(['molecule_chembl_id','pref_name','pchembl_value',
                'standard_type','canonical_smiles'])

        for a in acts:
            try:
                pc = float(a['pchembl_value'])
            except (TypeError, ValueError):
                continue
            if pc >= PCHEMBL_THRESHOLD:
                mol_name = (a.get('pref_name') or a.get('molecule_chembl_id') or '').lower()
                if any(ex in mol_name for ex in EXCLUDE):
                    continue
                results.append({
                    'gene': gene,
                    'target_id': tid,
                    'molecule_chembl_id': a['molecule_chembl_id'],
                    'molecule_name': a.get('pref_name',''),
                    'pchembl_value': pc,
                    'standard_type': a['standard_type'],
                    'smiles': a.get('canonical_smiles',''),
                })
    time.sleep(0.2)

chembl_df = pd.DataFrame(results)
print(f"Raw ChEMBL hits: {len(chembl_df)}")
chembl_df.to_csv(f"{DRIVE_DIR}/copd_chembl_raw.csv", index=False)

In [ ]:
# Cell 12 — USPTO novelty check
import requests, time

def check_uspto(mol_name, chembl_id):
    for query in [mol_name, chembl_id]:
        if not query:
            continue
        try:
            r = requests.get(
                'https://api.patentsview.org/patents/query',
                params={'q': f'{{"_text_phrase":{{"patent_abstract":"{query}"}}}}',
                        'f': '["patent_number","patent_title"]',
                        's': '[{"patent_date":"desc"}]',
                        'o': '{"per_page":1}'},
                timeout=15
            )
            if r.ok:
                data = r.json()
                if data.get('total_patent_count', 0) == 0:
                    return 'NOVEL_ALL'
                return f"PATENTS_FOUND:{data['total_patent_count']}"
        except Exception:
            pass
        time.sleep(0.3)
    return 'UNKNOWN'

if 'molecule_chembl_id' not in chembl_df.columns or len(chembl_df) == 0:
    print("No ChEMBL hits to check.")
else:
    unique_mols = chembl_df[['molecule_chembl_id','molecule_name']].drop_duplicates()
    print(f"Checking {len(unique_mols)} unique molecules against USPTO ...")
    novelty = {}
    for _, row in unique_mols.iterrows():
        novelty[row['molecule_chembl_id']] = check_uspto(
            row['molecule_name'], row['molecule_chembl_id']
        )
    chembl_df['novelty'] = chembl_df['molecule_chembl_id'].map(novelty)
    novel_df = chembl_df[chembl_df['novelty'] == 'NOVEL_ALL']
    print(f"NOVEL_ALL hits: {len(novel_df)}")
    chembl_df.to_csv(f"{DRIVE_DIR}/copd_chembl_novelty.csv", index=False)

In [ ]:
# Cell 13 — Scoring + ranking
# Score = N_targets x 10 + max_pChEMBL
scored = (
    chembl_df[chembl_df['novelty'] == 'NOVEL_ALL']
    .groupby('molecule_chembl_id')
    .agg(
        molecule_name  = ('molecule_name', 'first'),
        n_targets      = ('gene', 'nunique'),
        max_pchembl    = ('pchembl_value', 'max'),
        targets        = ('gene', lambda x: '|'.join(sorted(set(x)))),
        novelty        = ('novelty', 'first'),
        smiles         = ('smiles', 'first'),
    )
    .reset_index()
)
scored['priority_score'] = scored['n_targets'] * 10 + scored['max_pchembl']
scored = scored.sort_values('priority_score', ascending=False)

print("=== TOP 20 CANDIDATES ===")
print(scored[['molecule_name','molecule_chembl_id','n_targets',
              'max_pchembl','priority_score','targets']].head(20).to_string())

scored.to_csv(f"{DRIVE_DIR}/copd_candidates_scored.csv", index=False)

In [ ]:
# Cell 14 — Results summary + cross-disease signal check
CROSS_DISEASE_SIGNALS = [
    'WZ-3105','QL-XII-47','R-547','SAR405838','KPT-330',
    'CHEMBL5653589','MOLIBRESIB','CHEMBL3415598','DASATINIB'
]

print(f"\n{'='*60}")
print(f"COPD (GSE171541) — Top candidates summary")
print(f"{'='*60}")
print(f"DE genes (COPD vs ctl, padj<0.05):  {len(de)}")
print(f"ChEMBL hits (pChEMBL >= 7.5):       {len(chembl_df)}")
print(f"NOVEL_ALL:                           {len(novel_df)}")
print(f"\nTop 5:")
for _, row in scored.head(5).iterrows():
    print(f"  {row['molecule_name'] or row['molecule_chembl_id']:<30} "
          f"score={row['priority_score']:.1f}  targets={row['targets']}")

print(f"\nCross-disease signal check:")
top_names = scored['molecule_name'].str.upper().tolist()
for sig in CROSS_DISEASE_SIGNALS:
    hit = any(sig.upper() in n for n in top_names)
    if hit:
        print(f"  + {sig} — PRESENT")

In [ ]:
# Cell 15 — Save final checkpoint
FINAL_H5AD = f"{DRIVE_DIR}/copd_final.h5ad"

def save_checkpoint(adata, path):
    try:
        adata.write_h5ad(path)
        print(f"Saved: {path}")
        return True
    except Exception as e:
        print(f"WARNING: save failed — {e}")
        try:
            local = f"/content/copd_final_local.h5ad"
            adata.write_h5ad(local)
            print(f"Fallback save: {local}")
        except Exception as e2:
            print(f"WARNING: fallback also failed — {e2}")
        return False

save_checkpoint(adata, FINAL_H5AD)

In [ ]:
# Cell 16 — GitHub push
REPO = "copd-scrna"
import subprocess

cmds = [
    f"cd ~/repos/{REPO} && git add -A",
    f'cd ~/repos/{REPO} && git commit -m "COPD GSE171541 pipeline complete"',
    f"cd ~/repos/{REPO} && git push origin main",
]
for cmd in cmds:
    r = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True)
    print(r.stdout or r.stderr)

In [ ]:
# Cell 17 — Final summary print
print(f"""
╔══════════════════════════════════════════════════════╗
║  Patent 35 — COPD Pipeline Complete                 ║
║  Dataset:   GSE171541 ({adata.shape[0]:,} cells, {adata.shape[1]:,} genes)  ║
║  Clusters:  {adata.obs['leiden'].nunique()}                                      ║
║  TOP_CLUSTERS: {TOP_CLUSTERS}                        ║
║  Candidates: {len(scored)} scored, {len(novel_df)} NOVEL_ALL            ║
╚══════════════════════════════════════════════════════╝
""")